# 5. Stage 5: Data Rejoin

**Stage:** 5.3 of 5 — Daily Full Rejoin

**Purpose:** Same as Stage 5.1 but for the **full** dataset — includes all vacancy records regardless of whether they were re-posted on multiple days. Each record is matched across all available Stage 4 daily output files until a classified result is found.

**Input:**
- Stage 4 output pickles: `data/stage_04/processed/output/ua-YYYY-MM-DD.pkl`
- Stage 1 id/region/click pickles: `data/stage_01/intermediate/id_region_click/`
- Stage 4.5 region DB: `data/stage_04_5/processed/region_db.pkl`

**Output:** `data/stage_05/intermediate/pkl_daily_full/ua-YYYY-MM-DD.pkl` + matching `.json` and `_ua.json`

**Run:** Execute all cells top to bottom. No API calls required.

## 5.1. Description

The final stage of the processing pipeline consolidates all intermediate outputs into a single, enriched dataset. This unified format allows for structured analysis, filtering, and visualization of the Ukrainian job market across various dimensions such as skills, occupations, regions, and time.

## 5.2. Load packages and configuration

Enable autoreload, add `code/` to the Python path, and trigger memory cleanup.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append("../code")
import general as g
g.clean_memory()

Import `stage1` and `pandas`.

In [2]:
import stage1 as st1
import pandas as pd

Ensure all Stage 5 full output folders exist: PKL, JSON (all languages), and JSON (Ukrainian-only).

In [3]:
g.check_folder_exists(g.Config.STAGE5_DAILY_FULL_PKL_PATH)
g.check_folder_exists(g.Config.STAGE5_DAILY_FULL_JSON_PATH)
g.check_folder_exists(g.Config.STAGE5_DAILY_FULL_JSON_UA_PATH)

## 5.3. Get or create process log file

Define `get_stage5_process_df()` — same tracker pattern as Stage 5.1 but for the full dataset. Uses `STAGE5_PROCESS_FULL_PATH` as the tracker path.

In [4]:
def get_stage5_process_df(stage5_process_path, stage1_region_path):
    if not os.path.exists(stage5_process_path):
        columns = ["input_path", "region_path", "rejoin_path", "rejoin_status"]
        stage5_process = pd.DataFrame(columns=columns).astype(str)
        file_list = [f for f in os.listdir(stage1_region_path) if f]
        for file in file_list:
            path = os.path.join(stage1_region_path, file)
            new_row = pd.DataFrame([{'input_path': g.get_file_name_without_ext(file), 'region_path': path}])
            stage5_process = pd.concat([stage5_process, new_row], ignore_index=True)
        stage5_process = stage5_process.sort_values(by='input_path', ascending=True).reset_index(drop=True)
        stage5_process.to_pickle(stage5_process_path)
    else:
        stage5_process = pd.read_pickle(stage5_process_path)

    return stage5_process

Initialise or load the Stage 5 full process tracker.

In [5]:
process_df = get_stage5_process_df(g.Config.STAGE5_PROCESS_FULL_PATH, g.Config.STAGE1_ID_REGION_NCLICK_PATH)
process_df

,input_path,region_path,rejoin_path,rejoin_status
0,ua-2024-01-01,../data/stage_01/intermediate/id_region_click\...,NaN,NaN


Load the Stage 4.5 region lookup table and preview it to confirm the schema (`original`, `region`, `city`, `district`, `country`, `latitude`, `longitude`).

In [6]:
import pickle
region_final_db = pd.read_pickle(g.Config.STAGE4_5_REGION_DB_PATH)
region_final_db.head(2)

,original,city,district,region,country,latitude,longitude,is_remote
0,Odesa,Odesa,,Odesa Oblast,Ukraine,46.4825,30.7233,false
1,Sumy,Sumy,,Sumy Oblast,Ukraine,50.9077,34.7981,false


## 5.4. Rejoin data

### 5.4.1. Rejoin unique daily data

In this folder rejoined data with only daily published job vacancies, without prevoius day published

**Main rejoin loop** — for each daily file, enriches Stage 1 id/region/click data with lat/lon from `region_final_db`, then searches across all Stage 4 daily output files to find the ESCO classification for each vacancy. Unlike Stage 5.1 (unique only), this loop looks across multiple days to recover vacancies re-posted on different dates.

The production implementation is in the cell labelled `cols = [] / region_final_db = pd.read_pickle(...)` below. Earlier cells in this section are development iterations kept for reference.

In [7]:
region_final_db = pd.read_pickle(g.Config.STAGE4_5_REGION_DB_PATH)
process_df = pd.read_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)

In [8]:
cols = ['id', 'title', 'description', 'min_salary', 'max_salary', 'currency', 'salary_rate', 'date_created', 'date_expired', 'clean_title', 'clean_desc', 'title_lang', 'desc_lang', 'skill_ids', 'skill_labels','job_type', 'classified_code', 'classified_title', 'skill_labels_en', 'classified_title_clean', 'extract_type', 'esco_title', 'esco_skills', 'esco_id', 'esco_code',  'number_of_clicks', 'date', 'region_original', 'city', 'district', 'region', 'country', 'latitude', 'longitude']

In [9]:
region_final_db = pd.read_pickle(g.Config.STAGE4_5_REGION_DB_PATH)
process_df = pd.read_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)

for _, row in process_df.iterrows():

    if row["rejoin_status"] == "complete":
        if not os.path.exists(row["rejoin_path"]):
            print("File not found! AAAAAAAAAAAAAAAAAAAAAAAAAAALERT!")
        continue
    print(f'-------------------\n{_}.{row["input_path"]}')

    rejoin_done = None
    rejoin = None
    missing = None

    id_region_df = pd.read_pickle(row["region_path"])
    id_region_df['id'] = id_region_df['id'].astype(str)

    print(f'REGION ROWS -> {id_region_df.shape[0]}')

    merged_region = pd.merge(id_region_df, region_final_db, how='left', left_on='region', right_on='original')
    merged_region.drop(columns=['region_x'], inplace=True)
    merged_region.rename(columns={'original': 'region_original', 'region_y': 'region'}, inplace=True)
    merged_region['id'] = merged_region['id'].astype(str)

    day_path = os.path.join(g.Config.STAGE4_OUTPUT_PATH, f"{row["input_path"]}.pkl")

    if os.path.exists(day_path):
        stage4_df = pd.read_pickle(day_path)
        rejoin = merged_region.merge(stage4_df, on=["id"], how="left")
        rejoin_done = rejoin.dropna(subset=["clean_title"])
        missing = rejoin[rejoin["clean_title"].isnull()]
    else:
        rejoin_done = pd.DataFrame(columns=cols)
        rejoin = id_region_df
        missing = merged_region

    missing = missing.dropna(axis=1, how='all')

    if missing.shape[0] > 0:
        days = 0
        for __, _row in process_df.iterrows():
            day_path = os.path.join(g.Config.STAGE4_OUTPUT_PATH, f"{_row["input_path"]}.pkl")
            days +=1
            if not os.path.exists(day_path):
                continue

            stage4_df = pd.read_pickle(day_path)

            rejoin = missing.merge(stage4_df, on=["id"], how="left")

            if days % 100 == 0:
                print(f"{_row["input_path"]} - rejoin: {rejoin.shape[0]}")

            if rejoin.shape[0] > 0:
                done = rejoin.dropna(subset=["clean_title"])
                missing = rejoin[rejoin["clean_title"].isnull()]

                if done.shape[0] > 0:
                    rejoin_done = pd.concat([rejoin_done, done], ignore_index=True)
                if missing.shape[0] == 0:
                    continue

                missing = missing.dropna(axis=1, how='all')
                if missing.shape[0] == 0:
                    break

    pkl_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_FULL_PKL_PATH, f"{process_df.loc[_ ,"input_path"]}.pkl")
    rejoin_done.to_pickle(pkl_rejoin_path)

    json_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_FULL_JSON_PATH, f"{process_df.loc[_ ,"input_path"]}.json")
    rejoin_done.to_json(json_rejoin_path, orient='records')

    print(f'DONE ROWS -> {rejoin_done.shape[0]}')

    rejoin_done = rejoin_done[rejoin_done['country']=="Ukraine"]

    json_ua_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_FULL_JSON_UA_PATH, f"{process_df.loc[_ ,"input_path"]}.json")
    rejoin_done.to_json(json_ua_rejoin_path, orient='records')

    process_df.loc[_ ,"rejoin_path"] = pkl_rejoin_path
    process_df.loc[_ ,"rejoin_status"] = "complete"

    process_df.to_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)

-------------------
0.ua-2024-01-01
REGION ROWS -> 100
DONE ROWS -> 100


In [10]:
for _, row in process_df.iterrows():

    if row["rejoin_status"] == "complete":
        continue
    else:
        id_region_df = pd.read_pickle(row["region_path"])
        id_region_df['id'] = id_region_df['id'].astype(str)

        pickle_region = pd.merge(id_region_df, region_final_db, how='left', left_on='region', right_on='original')
        pickle_region.drop(columns=['region_x'], inplace=True)
        pickle_region = pickle_region.rename(columns={'original': 'region_original', 'region_y': 'region'})

        pickle_region['id'] = pickle_region['id'].astype(str)

        print(f"{_}-{row["input_path"]}-----------------")

        day_path = os.path.join(Config.STAGE4_OUTPUT_PATH, f"{row["input_path"]}.pkl")

        if os.path.exists(day_path):
            stage4_df = pd.read_pickle(day_path)
            rejoin = id_region_df.merge(clean_df, on=["id"], how="left")
            rejoin_done = rejoin.dropna(subset=["clean_title"])
            missing = rejoin[rejoin["clean_title"].isnull()]
        else:
            rejoin = id_region_df
            missing = rejoin

            if not os.path.exists(day_path):
                continue



    missing = missing.dropna(axis=1, how='all')

    if missing.shape[0] == 0:
        continue

    date = st1.extract_date_from_file_name(row["input_file"])

    for __, _row in process_stage_df[0:_].iterrows():

        if not os.path.exists(_row["clean_path"]):
            continue

        clean_df = pd.read_pickle(_row["clean_path"])

        rejoin = missing.merge(clean_df, on=["id"], how="left")
        #print(f"{_row["input_file"]} - rejoin: {rejoin.shape[0]}")

        if rejoin.shape[0] > 0:
            done = rejoin.dropna(subset=["clean_title"])

            missing = rejoin[rejoin["clean_title"].isnull()]
            if done.shape[0] > 0:
                rejoin_done = pd.concat([rejoin_done, done], ignore_index=True)
            if missing.shape[0] == 0:
                continue

            missing = missing.dropna(axis=1, how='all')

    key_cols = ["id", "region", "number_of_clicks", "date"]
    if missing.shape[0] > 0:
        id_region_df = id_region_df.merge(missing[key_cols], on=key_cols, how='left', indicator=True)
        id_region_df = id_region_df[id_region_df['_merge'] == 'left_only'].drop(columns=['_merge'])
        id_region_df.to_pickle(row["id_region_path"])
    print(f"FINAL REGION: {id_region_df.shape[0]}\n")


In [11]:
cols = []
region_final_db = pd.read_pickle(g.Config.STAGE4_5_REGION_DB_PATH)

for _file, _row in process_df.iterrows():
    if _row["rejoin_status"] == "complete":
        continue
    else:
        stage1_region = pd.read_pickle(process_df.loc[_file, "region_path"])
        stage1_region['id'] = stage1_region['id'].astype(str)

        pickle_region = pd.merge(stage1_region, region_final_db, how='left', left_on='region', right_on='original')
        pickle_region.drop(columns=['region_x'], inplace=True)
        pickle_region = pickle_region.rename(columns={'original': 'region_original', 'region_y': 'region'})

        pickle_region['id'] = pickle_region['id'].astype(str)

        print(f" - {pickle_region.shape[0]}")

        ready_data = pd.DataFrame(columns=pickle_region.columns)
        not_ready_data = pd.DataFrame(columns=pickle_region.columns)

        for __, __row in process_df.iterrows():
            day_path = os.path.join(g.Config.STAGE4_OUTPUT_PATH, f"{__row["input_path"]}.pkl")

            if not os.path.exists(day_path):
                continue

            daily_data = pd.read_pickle(day_path)
            daily_data['id'] = daily_data['id'].astype(str)

            drop_cols_list = daily_data.columns.tolist()[1:]

            if ready_data.empty:
                ready_data = pd.merge(pickle_region, daily_data, how='left', on='id')
                not_ready_data = ready_data[ready_data['esco_title'].isna()].copy()
                ready_data = ready_data[ready_data['esco_title'].notna()].copy()
                print(f"ready_start: {ready_data.shape[0]}")
            else:
                merged_daily_data = pd.merge(not_ready_data, daily_data, how='left', on='id')

                if 'esco_title' not in merged_daily_data.columns:
                    print("Warning: 'esco_title' column not found in merged data")
                    daily_ready_data = pd.DataFrame()
                else:
                    daily_ready_data = merged_daily_data[merged_daily_data['esco_title'].notna()].copy()

                if not daily_ready_data.empty:
                    ready_data = pd.concat([ready_data, daily_ready_data], ignore_index=True)
                    print(f"ready_next[{__row["input_path"]}]: {ready_data.shape[0]} - (+{daily_ready_data.shape[0]})")

                not_ready_data = merged_daily_data[merged_daily_data['esco_title'].isna()].copy()

            not_ready_data.drop(drop_cols_list, axis=1, inplace=True)

        #ready_data = pd.concat([ready_data, not_ready_data], ignore_index=True)

        print(f"{_file}. Day: {process_df.loc[_file ,"input_path"]} - not_ready {not_ready_data.shape[0]},s ready {ready_data.shape[0]} - ({stage1_region.shape[0]} vs {pickle_region.shape[0]})")

        pkl_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_FULL_PKL_PATH, f"{process_df.loc[_file ,"input_path"]}.pkl")
        ready_data.to_pickle(pkl_rejoin_path)

        json_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_FULL_JSON_PATH, f"{process_df.loc[_file ,"input_path"]}.json")
        ready_data.to_json(json_rejoin_path, orient='records')

        ready_data = ready_data[ready_data['country']=="Ukraine"]

        json_ua_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_FULL_JSON_UA_PATH, f"{process_df.loc[_file ,"input_path"]}.json")
        ready_data.to_json(json_ua_rejoin_path, orient='records')

        process_df.loc[_file ,"rejoin_path"] = pkl_rejoin_path
        process_df.loc[_file ,"rejoin_status"] = "complete"

        process_df.to_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)

print("Done")

YO


**Production implementation** — iterates over each daily file in the tracker. For each vacancy, searches all Stage 4 output files in chronological order until an `esco_title` is found. Vacancies unresolved after all files are checked are still written to output with `None` in ESCO columns. Saves PKL + JSON + Ukrainian JSON per day and updates the tracker.

In [14]:
region_final_db = pd.read_pickle(g.Config.STAGE4_5_REGION_DB_PATH)
stage5_files_list = sorted([f for f in os.listdir(g.Config.STAGE4_OUTPUT_PATH) if f])

cols = []

for _file, _row in process_df.iterrows():
    if _row["rejoin_status"] == "complete":
        continue
    else:
        stage1_region = pd.read_pickle(process_df.loc[_file, "region_path"])
        pickle_region = pd.merge(stage1_region, region_final_db, how='left', left_on='region', right_on='original')

        pickle_region.drop(columns=['region_x'], inplace=True)
        pickle_region = pickle_region.rename(columns={'original': 'region_original', 'region_y': 'region'})

        day_index = 0
        day_path = os.path.join(g.Config.STAGE4_OUTPUT_PATH, f"{_row["input_path"]}.pkl")
        print("-----------------------------")
        print(f"Working on ... {_row['input_path']} - day {_row['input_path']}")

        if os.path.exists(day_path):
            first_day = pd.read_pickle(day_path)
            if len(cols) == 0:
                cols = first_day.columns
        else:
            first_day = pd.DataFrame(cols)

        drop_cols_list = first_day.columns.tolist()[1:]
        pickle_region['id'] = pickle_region['id'].astype(str)
        first_day['id'] = first_day['id'].astype(str)

        pickle_region = pd.merge(pickle_region, first_day, how='left', on='id')
        ready_region = pickle_region[pickle_region['esco_title'].notna()].copy()
        missing_region = pickle_region[pickle_region['esco_title'].isna()].copy()

        missing_region.drop(drop_cols_list, axis=1, inplace=True)

        print(f"Working on ... {_row["input_path"]} - missing {missing_region.shape[0]}")
#and day_index <= _file
        while missing_region.shape[0] > 0 and day_index < process_df.shape[0] - 1:
            print(f"--- Working on ... {_row["input_path"]} - day {process_df.loc[day_index, "input_path"]} - missing {missing_region.shape[0]}")
            day_index += 1
            day_path = os.path.join(g.Config.STAGE4_OUTPUT_PATH, f"{process_df.loc[day_index, "input_path"]}.pkl")

            if os.path.exists(day_path):
                next_day = pd.read_pickle(day_path)
            else:
                continue

            next_day['id'] = next_day['id'].astype(str)
            pickle_region = pd.merge(missing_region, next_day, how='left', on='id')
            new_ready_region = pickle_region[pickle_region['esco_title'].notna()].copy()
            ready_region = pd.concat([ready_region, new_ready_region], ignore_index=True)
            print(f"{_file}. --- Working on ... {_row["input_path"]} - day {process_df.loc[day_index, "input_path"]} - ready {ready_region.shape[0]}")
            missing_region = pickle_region[pickle_region['esco_title'].isna()].copy()
            missing_region.drop(drop_cols_list, axis=1, inplace=True)

    ready_region = pd.concat([ready_region, missing_region], ignore_index=True)

    pkl_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_FULL_PKL_PATH, f"{process_df.loc[_file ,"input_path"]}.pkl")
    ready_region.to_pickle(pkl_rejoin_path)

    json_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_FULL_JSON_PATH, f"{process_df.loc[_file ,"input_path"]}.json")
    ready_region.to_json(json_rejoin_path, orient='records')

    ready_region = ready_region[ready_region['country']=="Ukraine"]

    json_ua_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_FULL_JSON_UA_PATH, f"{process_df.loc[_file ,"input_path"]}.json")
    ready_region.to_json(json_ua_rejoin_path, orient='records')

    process_df.loc[_file ,"rejoin_path"] = pkl_rejoin_path
    process_df.loc[_file ,"rejoin_status"] = "complete"
    print(f"Day: {process_df.loc[_file ,"input_path"]} - missing {missing_region.shape[0]} - ready {ready_region.shape[0]}")
    process_df.to_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)

process_df.to_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.